# Step 05 — LLM classification

Step 04 decided *which* buildings are places of activity. This notebook asks,
for each of them, **what activities happen inside and what kind of building it
is**, using an LLM on the evidence step 04 collected.

| | |
|---|---|
| **Reads** | `data/output/04_buildings_enriched.gpkg` — layer `buildings` (39,786 rows, 43 columns) and `building_pois` |
| **Writes** | `data/output/05_llm_input.parquet` — one record per building, the text the model reads, with the model-only columns beside it |
| **Needs** | `pyogrio`, `pandas`, `pyarrow`; the LLM client comes with section 3 |

## State of this notebook

Built **one step at a time**, each run and inspected before the next is written.

| step | | status |
|---|---|---|
| **05.1** | **The columns** — which the LLM reads, which only the model needs, which are noise for this step | **implemented** |
| **05.2** | **The prompt input** — one compact record per building from the 15 LLM columns, POI names paired with their uses, sources labelled | **implemented** |
| **05.3** | **The prompt and the output schema** — activity labels, Bosserhof class, confidence, reason | pending |
| **05.4** | **Routing** — per building where there is evidence, per signature where there is only class, land and size | pending |
| **05.5** | **The calls, and the validation against the rule baseline** | pending |

## Why an LLM, and what it is for

The previous pipeline settled this on an annotated set: the LLM reached 78.5 %
against 57.7 % for the rule table, and the whole difference was business-name
world knowledge — the model knows what *Deutsche Bank*, *Ernsting's family* or
*Tischlerei Holzteam* are. This step keeps that design and feeds the model what
step 04 has assembled per building: the POIs on it with names and uses, the
site around it, the cadastre class and name, the OSM footprint tag and name,
the land under it, and its size.

In [1]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError('Cannot find the pipeline root (the folder containing config.py). '
                       f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.')
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import time
import numpy as np
import pandas as pd
import pyogrio

from config import (
    ENRICHED_BUILDINGS_FILE, OUTPUT_DIR,
    LLM_COLUMN_ROLES, LLM_INPUT_COLS, LLM_MODEL_COLS, LLM_DROPPED_COLS,
    LLM_INPUT_FILE, LLM_RECORD_SAMPLE_PER_GROUP,
)
from lib.checks import require_file, require_non_empty, require_unique, require_cols
from lib.llm_record import build_records, GROUP_SAME_USE_FROM

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 70)

print('Root :', ROOT_DIR)
print('Input:', ENRICHED_BUILDINGS_FILE.name)

Root : C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
Input: 04_buildings_enriched.gpkg


## 1. The columns

The enriched layer carries 43 columns. Not all of them belong in a prompt, and
some must not be there. `LLM_COLUMN_ROLES` in `config.py` assigns every column
one of three roles, with the reason next to it; this section reads the layer,
checks that **every column is assigned** — a column added or removed in step 04
stops this step until it is classified — and shows what each one holds.

**The LLM sees** what describes what happens inside, in three blocks of
falling trust:

1. *what is inside* — `poi_uses`, `poi_names`, `site_uses`, `site_names`
2. *what the building is* — `label_en`, `name`, `osm_tag` (one field from
   `osm_twin_tag` on ALKIS rows and `osm_building` on OSM rows), `osm_twin_name`
3. *where and how big* — `alkis_landuse`, `alkis_landuse_detail` (the parcel's
   coded kind: education and science, health, power plant, campsite ...),
   `osm_landuse`, `city`, `area_m2`, `height_top_max_m`

**Only the model needs** the keys, the weight and the baseline: `building_id`,
`alkis_id`, `ags`, `function`, `volume_3d_m3`, `source`, `n_pois`, `n_sites`,
`address`, `activities`. Two of these are deliberately withheld from the
prompt. `activities` is the rule table's answer from the ALKIS class; shown, it
anchors the model to the rule, hidden, it is the baseline to validate against.
`address` proves a mailbox, not an activity, and the model has no lookup at
call time — asked about an address it would invent a tenant.

**Noise for this step** is step-03 provenance and QA, geometry bookkeeping,
columns already folded into others, the filter's own provenance (`rescued`,
`rescued_by`), and QGIS legend text. They stay in the step 04 output; they do
not enter step 05.

`alkis_landuse_detail` is the newest field, added 2026-09-14 after the size
floor left 6,052 buildings with nothing but class, land use and size. It is
the coded kind of the parcel under the building, one level below
`alkis_landuse`, translated through the AdV codelists in the GDI-DE registry
(see `ALKIS_LANDUSE_DETAIL_EN` in config, with the source URLs). A real case:
`DENIAL01000050Yi`, *Buildings for public purposes*, 787 m², five storeys, an
`office` footprint in OSM, no POI, no name. The class alone reads as
administration; the parcel says *education and science*, and the building
stands beside a state research institute. Filled on 7,493 kept buildings, and
on 821 of the 8,216 that have no POI, site or name — the group it was added for.

The three name columns stay separate on purpose. Measured on the layer: 1,345
buildings have an OSM footprint name that no POI carries, 1,521 an ALKIS name
and nothing else, and 10,466 have POI names but no footprint name. A name
repeated by three sources is a name three sources agree on; merged into one
list the model would not know who said what.

In [2]:
require_file(ENRICHED_BUILDINGS_FILE, 'enriched buildings (step 04)')
_layers = [l[0] for l in pyogrio.list_layers(ENRICHED_BUILDINGS_FILE)]
if 'buildings' not in _layers:
    raise AssertionError(f'layer "buildings" missing from {ENRICHED_BUILDINGS_FILE.name}: {_layers}')

print('Reading the enriched buildings (attributes only) ...', flush=True)
t0 = time.perf_counter()
bld = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='buildings', read_geometry=False)
print(f'  ok  {len(bld):,} buildings x {len(bld.columns)} columns  [{time.perf_counter() - t0:,.1f}s]')
require_non_empty(bld, 'enriched buildings')
require_unique(bld, 'building_id', 'enriched buildings')

# --- every column must have a role, and every role a column -------------------
_layer_cols = set(bld.columns)
_role_cols = set(LLM_COLUMN_ROLES)
_unassigned = sorted(_layer_cols - _role_cols)
_missing = sorted(_role_cols - _layer_cols)
if _unassigned or _missing:
    raise AssertionError(
        'config.LLM_COLUMN_ROLES and the layer disagree - '
        f'columns in the layer without a role: {_unassigned}; '
        f'roles for columns the layer no longer has: {_missing}. '
        'Classify them in config before running this step.')
print(f'  ok  all {len(_layer_cols)} columns have a role: '
      f'{len(LLM_INPUT_COLS)} the LLM sees, {len(LLM_MODEL_COLS)} model only, {len(LLM_DROPPED_COLS)} dropped here')

# --- what each column holds ------------------------------------------------------
def _example(s):
    v = s.dropna()
    if v.empty:
        return ''
    v = v.iloc[min(7, len(v) - 1)]
    return str(v)[:60]

review = pd.DataFrame({
    'role':      [LLM_COLUMN_ROLES[c][0] for c in bld.columns],
    'filled_%':  (bld.notna().mean() * 100).round(1).to_numpy(),
    'distinct':  bld.nunique().to_numpy(),
    'example':   [_example(bld[c]) for c in bld.columns],
    'why':       [LLM_COLUMN_ROLES[c][1] for c in bld.columns],
}, index=pd.Index(bld.columns, name='column'))
_order = {'llm': 0, 'model': 1, 'drop': 2}
review = review.iloc[np.argsort([_order[r] for r in review['role']], kind='stable')]
for _role, _title in (('llm', 'THE LLM SEES'), ('model', 'ONLY THE MODEL NEEDS'), ('drop', 'NOISE FOR THIS STEP - not read')):
    print()
    print(f'=== {_title} ({int((review["role"] == _role).sum())} columns)')
    print(review.loc[review['role'] == _role, ['filled_%', 'distinct', 'example', 'why']].to_string())

# --- how much of the layer has how much to say --------------------------------------
_has_inside = bld['poi_uses'].notna() | bld['site_uses'].notna()
_has_name = bld['name'].notna() | bld['osm_twin_name'].notna()
_tag = bld['osm_twin_tag'].fillna(bld['osm_building'])
_has_tag = _tag.notna() & (_tag != 'yes')
print()
print('  ..  what the LLM will have to work with, per building:')
print(f'        POIs or a site on it           {int(_has_inside.sum()):>7,}  ({100 * _has_inside.mean():.1f} %)')
print(f'        a name but no POI or site      {int((~_has_inside & _has_name).sum()):>7,}')
print(f'        only an informative OSM tag    {int((~_has_inside & ~_has_name & _has_tag).sum()):>7,}')
print(f'        class, land use and size only  {int((~_has_inside & ~_has_name & ~_has_tag).sum()):>7,}  '
      f'({100 * (~_has_inside & ~_has_name & ~_has_tag).mean():.1f} %) - the per-signature group, section 4')

  ok  04_buildings_enriched.gpkg (42.0 MB)
Reading the enriched buildings (attributes only) ...


  ok  39,786 buildings x 43 columns  [0.5s]
  ok  enriched buildings: 39,786 rows
  ok  enriched buildings.building_id: unique and non-null (39,786)
  ok  all 43 columns have a role: 15 the LLM sees, 10 model only, 18 dropped here

=== THE LLM SEES (15 columns)
                      filled_%  distinct                                             example                                                                                                                               why
column                                                                                                                                                                                                                        
name                      12.0      2528                                           Gärtnerei                                         the cadastre's own label (Tischlerei, Grundschule, Vereinsheim); OSM name on the gap rows
city                     100.0       134                             

## 2. The prompt input

One building, one record, one call. The model runs locally, so tokens cost
nothing but runtime — which is exactly why a record carries only what has
something to say, and says it compactly.

The record is a labelled block, not prose and not JSON. The previous pipeline
sent `key=value` pairs on two lines and reached 78.5 % on the annotated set; this
keeps that shape and changes four things:

* **POI names are paired with their uses** — `Star Tankstelle (fuel); Aral Shop
  (convenience)`. The building row's `poi_uses` and `poi_names` lists are *not*
  aligned (a building can carry four uses and three names), so the pairs come
  from the `building_pois` layer, ordered by share. Where many POIs share a use
  they are grouped under it — a snake farm with 20 `attraction` points reads
  `attraction x20: Königskobra; Netzpython; …` — so no name is lost and the
  record stays readable.
* **Every line names its source** — `inside`, `site`, `cadastre`, `osm
  footprint`, `land`, `place` — instead of a confidence level. The system prompt
  will say how much each source weighs; a name the cadastre gave stays
  distinguishable from one a mapper gave.
* **Empty lines are left out.** Absence is the information.
* **Numbers are rounded** to whole metres and square metres.

Two records as they will be sent. There is no id line: one building per call,
the answer is joined by position, and an id is only a string the model might
read something into. The `building_id` sits beside the record in the file.

```
cadastre: Buildings for public purposes
osm footprint: office
land: public facilities, education and science | osm: commercial
place: Braunschweig, Stadt | footprint 787 m2 | height 17 m
```

```
inside: Star Tankstelle (fuel)
cadastre: Gas station
osm footprint: yes
land: commercial services | osm: retail
place: Wolfsburg, Stadt | footprint 412 m2 | height 5 m
```

The rendering lives in `lib/llm_record.py`, importable on its own, so the
validation in 05.5 can reproduce exactly what the model saw. The output file
carries each record with the model-only columns beside it, and an `evidence`
column naming the group the building falls in: `poi_or_site`, `name_only`,
`tag_only`, `class_only`. Section 4 routes the last group per signature.

In [3]:
_layers = [l[0] for l in pyogrio.list_layers(ENRICHED_BUILDINGS_FILE)]
if 'building_pois' not in _layers:
    raise AssertionError(f'layer "building_pois" missing from {ENRICHED_BUILDINGS_FILE.name}: {_layers}')
pairs = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='building_pois', read_geometry=False)
require_cols(pairs, ['building_id', 'poi_role', 'poi_use', 'name', 'share_in_building', 'share_of_site'], 'building_pois')
_orphans = set(pairs['building_id']) - set(bld['building_id'])
if _orphans:
    raise AssertionError(f'{len(_orphans):,} building_pois rows point at buildings not in the layer, e.g. {sorted(_orphans)[:3]}')
print(f'  ok  {len(pairs):,} building-POI pairs on {pairs["building_id"].nunique():,} buildings '
      f'({int((pairs["poi_role"] != "site").sum()):,} own POIs, {int((pairs["poi_role"] == "site").sum()):,} site memberships)')

print('Rendering one record per building ...', flush=True)
t0 = time.perf_counter()
records = build_records(bld[list(LLM_INPUT_COLS) + ['building_id', 'source']], pairs)
print(f'  ok  {len(records):,} records  [{time.perf_counter() - t0:,.1f}s]')
require_unique(records, 'building_id', 'records')
if (records['n_chars'] < 40).any():
    raise AssertionError('a record came out nearly empty - every building has at least a class, land and size')

# --- what the model will read: size and shape --------------------------------------
print()
print('  ..  record length (characters):')
print(records['n_chars'].describe(percentiles=[.5, .9, .99]).round(0).astype(int).to_string())
_longest = records.sort_values('n_chars', ascending=False).head(3)
print('  ..  the three longest records belong to: '
      + ', '.join(f'{r.building_id} ({r.n_chars:,} chars)' for r in _longest.itertuples()))
print()
print('  ..  by evidence group:')
_g = records.groupby('evidence').agg(buildings=('building_id', 'size'), median_chars=('n_chars', 'median'))
print(_g.reindex(['poi_or_site', 'name_only', 'tag_only', 'class_only']).to_string())

# --- read some, one group at a time -----------------------------------------------------
_rng = np.random.default_rng(7)
for _grp in ('poi_or_site', 'name_only', 'tag_only', 'class_only'):
    _pool = records.index[records['evidence'] == _grp]
    print()
    print(f'=== {_grp} ({len(_pool):,} buildings) - {LLM_RECORD_SAMPLE_PER_GROUP} at random:')
    for _i in _rng.choice(_pool, size=min(LLM_RECORD_SAMPLE_PER_GROUP, len(_pool)), replace=False):
        print(records.at[_i, 'record'])
        print('-' * 60)
print()
print(f'=== the longest record, to see how grouping (same use x{GROUP_SAME_USE_FROM}+) keeps it readable:')
print(records.at[_longest.index[0], 'record'])

# --- write: the record next to the model-only columns ----------------------------------
llm_input = bld[list(LLM_MODEL_COLS)].merge(records, on='building_id', how='left')
if llm_input['record'].isna().any():
    raise AssertionError('a building has no record after the merge')
llm_input = llm_input[['building_id', 'evidence', 'record', 'n_chars'] + [c for c in LLM_MODEL_COLS if c != 'building_id']]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
llm_input.to_parquet(LLM_INPUT_FILE, index=False)
_back = pd.read_parquet(LLM_INPUT_FILE)
if len(_back) != len(bld) or list(_back.columns) != list(llm_input.columns):
    raise AssertionError('the parquet file does not read back as written')
print()
print(f'  ok  {len(_back):,} rows x {len(_back.columns)} columns -> {LLM_INPUT_FILE.name} '
      f'({LLM_INPUT_FILE.stat().st_size / 1e6:.1f} MB); columns: {list(_back.columns)}')

  ok  building_pois: has ['building_id', 'poi_role', 'poi_use', 'name', 'share_in_building', 'share_of_site']
  ok  41,285 building-POI pairs on 30,103 buildings (22,633 own POIs, 18,652 site memberships)
Rendering one record per building ...


  ok  39,786 records  [85.7s]
  ok  records.building_id: unique and non-null (39,786)

  ..  record length (characters):
count    39786
mean       208
std         49
min         93
50%        199
90%        264
99%        360
max       2710
  ..  the three longest records belong to: DENIAL0100005xF6 (2,710 chars), DENIAL9100005MxH (1,472 chars), DENIAL01000051gc (1,352 chars)

  ..  by evidence group:
             buildings  median_chars
evidence                            
poi_or_site      30103         210.0
name_only         1467         186.0
tag_only          2164         174.0
class_only        6052         162.0

=== poi_or_site (30,103 buildings) - 3 at random:
inside: KK Schießstand Almke (museum)
cadastre: Buildings for business or commerce
land: forestry
place: Wolfsburg, Stadt | footprint 30 m2 | height 2 m
------------------------------------------------------------
site: unnamed (commercial)
cadastre: Residential buildings with commercial and industrial properties
land: i


  ok  39,786 rows x 13 columns -> 05_llm_input.parquet (2.9 MB); columns: ['building_id', 'evidence', 'record', 'n_chars', 'alkis_id', 'ags', 'function', 'volume_3d_m3', 'source', 'n_pois', 'n_sites', 'address', 'activities']


## Where this leaves us

Sections 1 and 2 are done. The layer's 43 columns have a role in config, and
`05_llm_input.parquet` holds one record per building — the exact text the model
will read — with the model-only columns and the evidence group beside it. The
rendering is a library module, so the validation can reproduce the input.

### Next

**05.3, the prompt and the output schema.** The system prompt says what the
model is: a building activity interpreter that returns, as strict JSON, the
activity labels from the twelve-label list, one Bosserhof building-use class,
a confidence of low / medium / high, and one sentence of reasoning that names
the evidence used. It also says how the sources weigh: what is inside first,
the cadastre and the OSM footprint second, land and size as context — size sets
the scale of an activity, not its kind. Both texts come to you before any call
is made.

**05.4, routing.** Buildings with evidence go to the model one by one. The
`class_only` group carries nothing but class, land use and size and is
classified once per distinct combination of those, the answer copied onto every
member and marked as such.

**05.5, the calls and the validation** against the rule baseline in
`activities`, on the annotated Bosserhof set of the previous pipeline.